# 13 · Library-free equation discovery with neural jets

`omnibias-symbolic` turns omnibias's exact activation derivative towers into a
scientific-discovery engine. The key trick: because every activation exposes a
closed-form n-th derivative `σ⁽ⁿ⁾`, we can build the full derivative *jet*
`y, dy, d2y, …` of a fitted field **exactly**, and then search for a compact
*implicit* relation among those generic coordinates — with **no named** `sin`,
`exp`, or `tanh` in the library.

This notebook recovers four kinds of closed-form law:
1. differential identities of activations (`dy = y`, `d2y = -y`, `dy = 1 - y²`),
2. an interpretable AutoML surrogate of a hidden multivariate law,
3. a PDE operator coefficient (the heat equation), and
4. the Blasius boundary-layer identity `f''' = -½ f f''`.

In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from _style import set_style, PRIMARY, ACCENT, GOOD
set_style()

from omnibias.symbolic import (
    discover_activation_identity,
    discover_interpretable_surrogate,
    discover_pde_operator_law,
    make_symbolic_regression_dataset,
)
from omnibias.symbolic.blasius import solve_blasius, discover_blasius_identity

## 1. Differential identities, no named functions

We represent an activation exactly as a one-neuron field, extract its closed-form
jet, and let `NeuralJetDiscoverer` find the lowest-complexity relation. It returns
the textbook identities to machine precision.

In [ ]:
cases = [("exp", (-1.0, 1.0), (1,)), ("sin", (-np.pi, np.pi), (2,)), ("tanh", (-1.0, 1.0), (1,))]
rows = []
for act, rng, lhs in cases:
    r = discover_activation_identity(act, x_range=rng, candidate_lhs_orders=lhs)
    rows.append((act, r.formula(), r.test_rmse))
    print(f"{act:>5}:  {r.formula():<16}  test RMSE = {r.test_rmse:.2e}")

fig, ax = plt.subplots(figsize=(7.0, 3.6))
ax.bar([a for a, _, _ in rows], [e for _, _, e in rows], color=PRIMARY)
ax.set_yscale("log")
ax.set_ylabel("test RMSE (log)")
ax.set_title("Recovered identities:  exp→dy=y   sin→d2y=-y   tanh→dy=1-y²")
for i, (_, f, _) in enumerate(rows):
    ax.text(i, rows[i][2], f, ha="center", va="bottom", fontsize=10)
plt.tight_layout()

## 2. AutoML surrogate of a hidden multivariate law

Hidden law: `y = 1.5 x₁² - 2 x₂x₃ + sin(2x₄) + 0.4 cos(x₄)`. The discoverer
selects among Taylor / Fourier / hybrid libraries on a validation split and
recovers every active term with the right coefficient.

In [ ]:
data = make_symbolic_regression_dataset(n_samples=900, noise_std=0.0, seed=0)
surrogate = discover_interpretable_surrogate(data, complexity_weight=5e-4)
print("selected family:", surrogate["family"])
print("equation:       ", surrogate["equation"])

true = {"x1^2": 1.5, "x2*x3": -2.0, "sin(2*x4)": 1.0, "cos(x4)": 0.4}
found = {row["name"]: row["coefficient"] for row in surrogate["selected_terms"]}
names = list(true)
x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.bar(x - 0.2, [true[n] for n in names], 0.4, label="true", color=PRIMARY)
ax.bar(x + 0.2, [found.get(n, 0.0) for n in names], 0.4, label="discovered", color=ACCENT)
ax.set_xticks(x); ax.set_xticklabels(names)
ax.axhline(0, color="#888", lw=1)
ax.set_title("Recovered coefficients vs ground truth")
ax.legend(); plt.tight_layout()

## 3. PDE operator coefficient and the Blasius identity

From exact derivative columns of a two-mode heat field, the sparse fit recovers
`u_t = 0.12 u_xx`. The Blasius boundary layer (no elementary closed form) is
solved by shooting, and its governing identity `f''' = -½ f f''` is recovered
from the numerical jets.

In [ ]:
pde = discover_pde_operator_law(diffusivity=0.12)
print("PDE:", pde["equation"], "  RMSE =", f"{pde['metrics']['rmse']:.2e}")

sol = solve_blasius()
blas = discover_blasius_identity(sol)
print("Blasius f''(0) =", f"{sol.fpp0:.6f}", " identity:", blas["equation"],
      "  test RMSE =", f"{blas['metrics']['test_rmse']:.2e}")

fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.plot(sol.eta, sol.f, color=PRIMARY, label="f")
ax.plot(sol.eta, sol.fp, color=ACCENT, label="f' (velocity)")
ax.plot(sol.eta, sol.fpp, color=GOOD, label="f''")
ax.axhline(1.0, color="#888", lw=1, ls="--")
ax.set_xlabel("η"); ax.set_title("Blasius boundary layer (shooting), f'(∞)→1")
ax.legend(); plt.tight_layout()

## Takeaway

Closed-form omnibias jets make equation discovery *library-free*: the same engine
recovers activation identities, a multivariate symbolic surrogate, a PDE
coefficient, and the Blasius law — each to near machine precision — without being
handed the answer's functional form. See `examples/symbolic_discovery/` for the
applied benchmarks (battery, turbofan, finance) built on the same idea.